# Nucleolar protein distribution quantification
In this notebook, we show how we quantify the distribution of nucleolar proteins- Fbl and Npm1 around the nucleolus in speckle KD (SON KD, SRRM2 KD and Double KD) and Ubtf rescue conditions (SON KD + Ubtf1 mRNA).
In brief, we segment the nucleus and nucleolus using the DAPI channel. We use speckle marker proteins (if available) to improve nucleoli segmentation (speckle protein dense regions are not nucleoli but have low DAPI intensity, which can lead to false positive nucleolus). Then, we make concentric ring around and the nucleolus and measure the mean Fbl/Npm1 intensity inside these rings.

In [ ]:
import fishtools as ft
import warnings
import numpy as np
import os
import pandas as pd
from aicsimageio import AICSImage
import napari
from skimage import filters, morphology, measure, feature, segmentation, transform, draw, exposure, ndimage
from scipy import ndimage
import re
import matplotlib.pyplot as plt
import seaborn as sns
import tifffile

# functions

In [46]:
def nucleoli_table(nuc_labels, nucl_labels, scale=(1.0, 1.0, 1.0), filename="image.tif"):
    voxel_vol = float(np.prod(scale))  # Physical volume of one voxel.

    # Volume (and label) of every nucleolus.
    props = pd.DataFrame(measure.regionprops_table(nucl_labels, properties=("label", "area")))
    props = props.rename(columns={"label": "nucleolus_id", "area": "voxels"})
    props["nucleolus_volume"] = props["voxels"] * voxel_vol

    # Majority-overlap parent: count (nucleolus, nucleus) voxel pairs in one pass.
    m = nucl_labels > 0
    pairs = np.stack([nucl_labels[m].ravel(), nuc_labels[m].ravel()], axis=1)
    counts = (pd.DataFrame(pairs, columns=["nucleolus_id", "nucleus_id"])
                .value_counts()
                .rename("overlap")
                .reset_index())

    # Keep only the nucleus with the largest overlap for each nucleolus.
    best = (counts.sort_values("overlap", ascending=False)
                  .drop_duplicates("nucleolus_id"))

    df = props.merge(best[["nucleolus_id", "nucleus_id"]], on="nucleolus_id", how="left")
    df["nucleus_id"] = df["nucleus_id"].fillna(0).astype(int)  # 0 = no parent (background overlap only).
    df.insert(0, "filename", filename)
    return df[["filename", "nucleus_id", "nucleolus_id", "nucleolus_volume"]]

In [47]:
def physical_ball_footprint(scale_zyx, radius_um):  # Build a physically spherical footprint for a (z, y, x) array.
    spacing = np.asarray(scale_zyx, dtype=float)  # Convert voxel spacing to a numeric (z, y, x) array.
    if spacing.shape != (3,) or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):  # Check that spacing has three positive finite values.
        raise ValueError("scale_zyx must contain three positive finite values.")  # Explain invalid spacing.
    if not np.isfinite(radius_um) or radius_um < 0:  # Check that the radius is finite and nonnegative.
        raise ValueError("radius_um must be a finite, nonnegative value.")  # Explain invalid radius.
    extents = np.ceil(radius_um / spacing).astype(int)  # Find the voxel extent needed on each axis.
    z, y, x = np.ogrid[-extents[0]:extents[0] + 1, -extents[1]:extents[1] + 1, -extents[2]:extents[2] + 1]  # Make broadcastable voxel-coordinate grids.
    distance_squared = (z * spacing[0])**2 + (y * spacing[1])**2 + (x * spacing[2])**2  # Compute squared distance in microns.
    return distance_squared <= radius_um**2  # Return the Boolean footprint inside the physical sphere.

In [ ]:
def signal_vs_distance_v2(S1, S2, N, P, spacing=None, delta=0.1): 
    """Mean signal in shells of width `delta` around the nucleoli, inside the nucleus.

    S1       : signal image 1 (2D or 3D), values >= 0
    S2       : signal image 2 (2D or 3D), values >= 0
    N, P    : nucleus and nucleoli masks, same shape as S (nonzero = inside)
    spacing : voxel size per axis, e.g. (dz, dy, dx) in µm; None = pixel units
    delta   : shell width, in the same units as spacing

    Returns (outer edge of each shell, mean signal per shell, voxel count per shell).
    Empty shells have mean NaN and count 0.
    """
    N, P = np.asarray(N) > 0, np.asarray(P) > 0
    if not (S1.shape == N.shape == P.shape == S2.shape):
        raise ValueError("S, N and P must have the same shape")
    V = N & ~P                                                # nucleoplasm
    if not P.any() or not V.any():
        raise ValueError("need at least one nucleolus pixel and one nucleoplasm pixel")
    
    E=get_edge_mask(P)

    ED = ndimage.distance_transform_edt(~E, sampling=spacing)  # distance to nearest nucleolus edge
    R = np.where(P,-(np.ceil(ED / delta).astype(int)),np.ceil(ED / delta).astype(int))
    R = np.where(E,0,R)

    shells = np.arange(R[N].min(), R[N].max() + 1)
    background_label = shells[0] - 1
    labels = np.where(N, R, background_label)                                # 0 = background/ignored

    SIG1 = ndimage.mean(S1, labels=labels, index=shells)
    SIG2 = ndimage.mean(S2, labels=labels, index=shells)
    K = ndimage.sum(N, labels=labels, index=shells).astype(int)
    return shells * delta, SIG1, SIG2, K, labels

In [49]:
def process_path_shells(path, delta=0.5, min_hole_um3=1000, min_size_nucleoli=1000,
                        footprint_radius=0.7, chromatin_sigma=1, use_speckles=True):
    """Run full segmentation + shell-intensity analysis on a single tif path."""
    file = os.path.basename(path)
    condition = path.split("_")[0]

    channels, scale = ft.get_data_aicsimageio_v2(path)
    sonlayer, fiblayer, dapilayer, specklelayer = channels

    # Segmentation
    nuclei_labels, nuclei_mask, num_nuclei_unf = ft.segment_nucleus(
        dapilayer, scale_zyx=scale, remove_edge_nuclei=False, otsu_multiplier=1
    )
    nuclei_labels_eroded = morphology.binary_erosion(nuclei_labels, footprint=physical_ball_footprint(scale,footprint_radius))
    chromatin_mask = ft.mask_chromatin(dapilayer, scale_zyx=scale, nuclei_mask=nuclei_mask, min_hole_um3=min_hole_um3, sigma=chromatin_sigma, otsu_multiplier=0.7)

    # Use the speckle channel if available to rule out nucleoli.
    if use_speckles:
        sigma=1
        ratio_zy=scale[0]/scale[1]
        voxel_volume=np.prod(scale)
        volume_nuclei_mask=nuclei_mask*specklelayer #volume is the speckle channel
        sigma_tuple=(sigma*(1/ratio_zy),sigma,sigma)
        volume_blur=filters.gaussian(volume_nuclei_mask, sigma=sigma_tuple, mode='nearest', preserve_range=True)
        triangle_thresh=filters.threshold_triangle(volume_blur[nuclei_mask]) #threshold only on pixels inside the nucleus
        speckle_mask=volume_blur>triangle_thresh

        chromatin_mask2= chromatin_mask | speckle_mask
        nucleoli_mask = nuclei_labels_eroded * ~chromatin_mask2
    else:
        nucleoli_mask = nuclei_labels_eroded * ~chromatin_mask

    nucleoli_mask_opened = morphology.opening(nucleoli_mask,footprint=physical_ball_footprint(scale,0.7))
    nucleoli_small_removed = morphology.remove_small_objects(nucleoli_mask_opened, min_size=min_size_nucleoli)

    nucleoli_labels, num_labels=ndimage.label(nucleoli_small_removed)

    # Shell intensity analysis
    shells, signal_1, signal_2 , voxeln, labels = signal_vs_distance_v2(fiblayer, dapilayer, nuclei_mask, nucleoli_labels, spacing=scale, delta=delta)

    df_shell = pd.DataFrame({
        "path": path,
        "file": file,
        "condition": condition,
        "shell_um": shells,
        "sig1_mean_intensity": signal_1,
        "sig2_mean_intensity": signal_2,
        "voxel_count": voxeln
    })

    df_nucleoli = nucleoli_table(nuclei_labels,nucleoli_labels,scale=scale,filename=file)

    return df_shell, df_nucleoli

In [ ]:
def get_edge_mask(mask, footprint=morphology.ball(1)):
    """Returns a mask of just the boundary (edge) voxels of the input mask."""
    mask = mask.astype(bool)
    eroded = morphology.binary_erosion(mask, footprint=footprint)
    edge = mask & ~eroded
    return edge

## Parameter optimization
Here we try to find the right parameters for nucleus, nucleoli segmentation of the images one-by-one and trouble-shoot.

In [ ]:
tif_list=ft.find_czi_files("/Users/agastya.singh/Desktop/julias nucleolus staining KD/fibrillarin",None)

In [ ]:
path=tif_list[22]
print(path)

In [ ]:
condition=path.split("_")[0]
cell_stage="4c"
file=os.path.basename(path)
use_speckles=True

#Close if another viewer is open
try:
    if viewer is not None:
        viewer.close()
except Exception:
    pass

viewer=ft.view_napari(path,c=4,title=file)

#This is where you assign the correct channels to the images. Replade fiblayer with npmlayer for npm images. Make sure the order is correct.
sonlayer=viewer.layers[0].data
fiblayer=viewer.layers[1].data
dapilayer=viewer.layers[2].data
specklelayer=viewer.layers[3].data
scale,ratio=ft.get_scale(viewer.layers[0])

nuclei_labels,nuclei_mask,num_nuclei_unf=ft.segment_nucleus(dapilayer,scale_zyx=scale,remove_edge_nuclei=True, otsu_multiplier=1,sigma=6)
nuclei_labels_eroded = morphology.binary_erosion(nuclei_labels, footprint= physical_ball_footprint(scale,0.7))

chromatin_mask=ft.mask_chromatin(dapilayer,scale_zyx=scale,nuclei_mask=nuclei_mask,min_hole_um3=100,sigma=1,otsu_multiplier=0.7)

if use_speckles:
    sigma=1
    ratio_zy=scale[0]/scale[1]
    voxel_volume=np.prod(scale)
    volume_nuclei_mask=nuclei_mask*specklelayer #volume is the speckle channel
    sigma_tuple=(sigma*(1/ratio_zy),sigma,sigma)
    volume_blur=filters.gaussian(volume_nuclei_mask, sigma=sigma_tuple, mode='nearest', preserve_range=True)
    triangle_thresh=filters.threshold_triangle(volume_blur[nuclei_mask]) #threshold only on pixels inside the nucleus
    speckle_mask=volume_blur>triangle_thresh

    chromatin_mask2= chromatin_mask | speckle_mask
    nucleoli_mask = nuclei_labels_eroded * ~chromatin_mask2
else:
    nucleoli_mask = nuclei_labels_eroded * ~chromatin_mask


nucleoli_mask_opened = morphology.opening(nucleoli_mask,footprint=physical_ball_footprint(scale,0.7))
nucleoli_small_removed = morphology.remove_small_objects(nucleoli_mask_opened, min_size=3000)
nucleoli_labels, num_labels=ndimage.label(nucleoli_small_removed)

In [ ]:
viewer.add_labels(nuclei_labels,scale=scale)

if use_speckles:
    viewer.add_labels(chromatin_mask2,scale=scale)
else:
    viewer.add_labels(chromatin_mask,scale=scale)

viewer.add_labels(nucleoli_mask,scale=scale)
viewer.add_labels(nucleoli_mask_fillholes,scale=scale)
viewer.add_labels(nucleoli_mask_opened,scale=scale)
viewer.add_labels(nucleoli_labels,scale=scale)
viewer.add_labels(speckle_mask,scale=scale)
viewer.add_image(dapilayer,scale=scale)

## Batch loop for processing all the images with the same paramters after optimization

In [ ]:
czi_list=ft.find_czi_files("/Users/agastya.singh/Desktop/julias nucleolus staining KD/fibrillarin",None)

In [ ]:
# ------------------ Batch loop over all paths ------------------
all_dfs_shell = []
all_dfs_nucleoli = []

for i, path in enumerate(tif_list):
    print(f"[{i+1}/{len(tif_list)}] Processing: {path}")
    try:
        df_shell, df_nucleoli = process_path_shells(path,delta=0.3, min_hole_um3=100, footprint_radius=0.7, chromatin_sigma=1)
        all_dfs_shell.append(df_shell)
        all_dfs_nucleoli.append(df_nucleoli)
    except Exception as e:
        print(f"  FAILED on {path}: {e}")

shell_df = pd.concat(all_dfs_shell, ignore_index=True)
nucleoli_df = pd.concat(all_dfs_nucleoli, ignore_index=True)

In [ ]:
shell_df['condition']=shell_df.file.str.split(".czi").str[0]
shell_df['sig1_normalized_intensity_'] = shell_df.groupby('condition')['sig1_mean_intensity'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
shell_df['sig1_normalized_intensity_perfile'] = shell_df.groupby(['condition','file'])['sig1_mean_intensity'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
shell_df['sig1_zscore_intensity'] = shell_df.groupby('condition')['sig1_mean_intensity'].transform(
    lambda x: (x - x.mean()) / x.std()
)
shell_df['sig2_normalized_intensity_'] = shell_df.groupby('condition')['sig2_mean_intensity'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
shell_df['sig2_normalized_intensity_perfile'] = shell_df.groupby(['condition','file'])['sig2_mean_intensity'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)
shell_df['sig2_zscore_intensity'] = shell_df.groupby('condition')['sig2_mean_intensity'].transform(
    lambda x: (x - x.mean()) / x.std()
)
shell_df

In [ ]:
shell_df.to_csv("/Users/agastya.singh/Desktop/julias nucleolus staining KD/fibrillarin/Fbl_aroundnucleoliintensity_quants_speckletrue.tsv",sep="\t",header=True,index=False)

In [ ]:
nucleoli_df.to_csv("/Users/agastya.singh/Desktop/julias nucleolus staining KD/fibrillarin/Fbl_nucleolitable_speckletrue.tsv",sep="\t",header=True,index=False)